# Lab 4

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn

In [4]:
# For this laboratory, there are different datasets that can be used.
# For now, we will use the raw_train dataset, which is the training set without any preprocessing.

path = "./data/raw_train.csv"
path_summary = "./data/train_summary.csv"

df = pd.read_csv(path)
df_summary = pd.read_csv(path_summary)

## Data Cleaning

### Data Cleaning on the Raw Training Set

Get information about the dataset

In [ ]:
df.head(400)

In [ ]:
df.info()

There are some null values on the unit_english_id column, which is 2 only rows. We can drop those rows without losing any information.

In [ ]:
print(df["unit_english_id"].isnull().sum())

# Drop the rows with null values in the unit_english_id column
df = df.dropna(subset=["unit_english_id"])

In [ ]:
df.info()

The data for the bid_date is in object. Parse to date

In [ ]:
df["bid_date"] = pd.to_datetime(df["bid_date"])

df.info()

Lets check for possible duplicates in the dataset. We can drop them if there are any.

In [ ]:
df.duplicated().sum()

Duplicates found. Drop them.

In [ ]:
df.drop_duplicates(inplace=True)

Verify that the duplicates have been dropped.

In [ ]:
df.duplicated().sum()

Another thing that we can check is to verify that the total_bid (the target variable) is not negative and if it is the result of multipying the quantity and the amount. If so, then there is a leakage in the data, and we should drop the total_bid column and use the quantity and amount to calculate the total_bid when needed.

In [ ]:
(df["quantity"] * df["amount"] != df["total_bid"]).sum()

## Feature Engineering

Create new columns based on the columns we have.

Separate the bid_date column into year, month and day columns.

In [ ]:
df["bid_month"] = df["bid_date"].dt.month
df["bid_year"] = df["bid_date"].dt.year
df["bid_day"] = df["bid_date"].dt.dayofweek

Have aggregate based features

In [ ]:
# Contractor-based analysis:
# total number of bids per contractor
df["contractor_bid_count"] = df.groupby("contractor_id")["row_id"].transform(
    "nunique"
)
# average bid amount per contractor
df["contractor_avg_quantity"] = df.groupby("contractor_id")["quantity"].transform(
    "mean"
)
# average bid amount per contractor
df["contractor_avg_amount"] = df.groupby("contractor_id")["amount"].transform(
    "mean"
)

# Unit-based analysis:
# total number of bids per unit
df["unit_bid_count"] = df.groupby("unit_english_id")["row_id"].transform("nunique")
# average bid amount per unit
df["unit_avg_quantity"] = df.groupby("unit_english_id")["quantity"].transform(
    "mean"
)
# average bid amount per unit
df["unit_avg_amount"] = df.groupby("unit_english_id")["amount"].transform("mean")

# contractor's bid frequency this year
# df["contractor_bids_this_year"] = df.groupby(
#     ["contractor_id", "bid_year"]
# )["row_id"].transform("nunique")

Preview that the new features have been created correctly.

In [ ]:
df.head(15)

## Visualization

In [ ]:
# Check on what ranges does the total_bid variable lie
sns.histplot(df["total_bid"], kde=True)
plt.title("total_bid distribution")
plt.show()

print(df["total_bid"].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]))

In [ ]:
# How often does a contractor bid in a year?
# sns.histplot(df["contractor_bids_this_year"], kde=True)
# plt.title("contractor_bids_this_year distribution")
# plt.show()

In [ ]:
# What is the total number of bids per contractor?
sns.histplot(df["contractor_bid_count"], kde=True)
plt.title("contractor_bid_count distribution")
plt.show()

In [ ]:
# How many bids are there per unit?
sns.countplot(x=df["unit_english_id"])
plt.title("Number of bids per unit")
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Time-based analysis: per year and per month
df_time = df.copy()
df_time["bid_date"] = pd.to_datetime(df_time["bid_date"], errors="coerce")
df_time = df_time.dropna(subset=["bid_date"])

# Yearly aggregates
df_yearly = (
    df_time.assign(bid_year=df_time["bid_date"].dt.year)
    .groupby("bid_year", as_index=False)
    .agg(
        avg_total_bid=("total_bid", "mean"),
        median_total_bid=("total_bid", "median"),
        bid_count=("row_id", "count"),
    )
)

# Per-year visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(data=df_yearly, x="bid_year", y="bid_count", ax=axes[0], color="steelblue")
axes[0].set_title("Bids per Year")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Bid Count")

sns.lineplot(
    data=df_yearly, x="bid_year", y="median_total_bid", marker="o", ax=axes[1]
)
axes[1].set_title("Median total_bid per Year")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Median total_bid")

plt.tight_layout()
plt.show()

In [ ]:
# Monthly aggregates
df_monthly = (
    df_time.set_index("bid_date")
    .resample("MS")
    .agg(
        avg_total_bid=("total_bid", "mean"),
        median_total_bid=("total_bid", "median"),
        bid_count=("row_id", "count"),
    )
    .reset_index()
)

# Per-month visualizations
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

sns.lineplot(data=df_monthly, x="bid_date", y="bid_count", marker="o", ax=axes[0])
axes[0].set_title("Bids per Month")
axes[0].set_xlabel("")
axes[0].set_ylabel("Bid Count")

sns.lineplot(
    data=df_monthly, x="bid_date", y="median_total_bid", marker="o", ax=axes[1]
)
axes[1].set_title("Median total_bid per Month")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Median total_bid")

plt.tight_layout()
plt.show()

In [ ]:
# Per-month aggregates (month-of-year, across all years)
df_monthly_seasonal = (
    df_time.assign(bid_month=df_time["bid_date"].dt.month)
    .groupby("bid_month", as_index=False)
    .agg(
        bid_count=("row_id", "count"),
        median_total_bid=("total_bid", "median"),
    )
)

month_map = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec",
}
df_monthly_seasonal["month_name"] = df_monthly_seasonal["bid_month"].map(month_map)

# Per-month visualizations (seasonality across all years)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(
    data=df_monthly_seasonal,
    x="month_name",
    y="bid_count",
    ax=axes[0],
    color="slateblue",
)
axes[0].set_title("Total Bids by Month (All Years)")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Bid Count")

sns.lineplot(
    data=df_monthly_seasonal,
    x="month_name",
    y="median_total_bid",
    marker="o",
    ax=axes[1],
)
axes[1].set_title("Median total_bid by Month (All Years)")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Median total_bid")

plt.tight_layout()
plt.show()

## Preprocessing

### Drop Unnecessary Columns

Drop identifier columns (`row_id`, `job_id`, `pay_item_id`) and `bid_date` (already decomposed into `bid_month`, `bid_year`, `bid_day`). Also drop `category_description` since it is redundant with the numeric `category_id`, and drop `contractor_id` since it has been encoded into aggregate features.

In [ ]:
cols_to_drop = ["row_id", "job_id", "pay_item_id", "bid_date", "category_description", "contractor_id"]

df_model = df.drop(columns=cols_to_drop)

print(f"Shape after dropping: {df_model.shape}")
df_model.dtypes

### Encode Categorical Features

Use one-hot encoding (`pd.get_dummies`) on the remaining nominal columns (`job_category_description`, `pay_item_description`, `unit_english_id`, `primary_location`).

In [ ]:
cat_cols = ["job_category_description", "pay_item_description", "unit_english_id", "primary_location"]

# Do not include pay_item_description in the OHE as it has a very high cardinality (over 1000 unique values)

cat_cols_to_ohe = [col for col in cat_cols if col != "pay_item_description"]

df_encoded = pd.get_dummies(df_model, columns=cat_cols_to_ohe, drop_first=True)

print(f"Shape before OHE : {df_model.shape}")
print(f"Shape after OHE  : {df_encoded.shape}")
print(f"New columns added: {df_encoded.shape[1] - df_model.shape[1]}")
df_encoded.dtypes

In [ ]:
df_encoded.head(20)

### Define Features (X) and Target (y)

The target variable is `total_bid`. All remaining columns serve as features. The `amount` and `quantity` columns are kept as features since they are individual pay-item level values and do not directly equal `total_bid` at the row level (no leakage confirmed earlier).

In [ ]:
TARGET = "total_bid"

X = df_encoded.drop(columns=[TARGET])
y = df_encoded[TARGET]

print(f"Features shape : {X.shape}")
print(f"Target shape   : {y.shape}")
print(f"\nFeature columns:\n{X.columns.tolist()}")

### Train / Test Split (80 / 20)

Split the data before any fit-based preprocessing (scaling) to prevent data leakage. Using `shuffle=True` to avoid ordering bias and `random_state=42` for reproducibility.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")